# Archived research notebook

Original filename: `preprocessing seventh通道选择.ipynb`  
Purpose: Select EEG channels using the research workflow.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split
import random
import math
from tqdm import tqdm
import re

# 设置matplotlib使用不需要中文字体的配置
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['axes.unicode_minus'] = False


class QPSOParticle:
    """Quantum-inspired Particle Swarm Optimization particle"""

    def __init__(self, n_channels, n_select=12):
        self.position = np.zeros(n_channels)
        # Initialize with some random channels selected
        selected_indices = np.random.choice(n_channels, n_select, replace=False)
        self.position[selected_indices] = 1
        self.pbest = self.position.copy()
        self.pbest_fitness = -float('inf')

    def update_position(self, gbest, beta):
        # Quantum-inspired position update
        for i in range(len(self.position)):
            # Calculate quantum probability
            p = random.random()
            if p < 0.5:
                self.position[i] = 1 if random.random() < abs(beta * self.pbest[i] + (1 - beta) * gbest[i]) else 0
            else:
                self.position[i] = 1 if random.random() < abs(beta * gbest[i] + (1 - beta) * self.pbest[i]) else 0

        # Ensure exactly n_select channels are chosen
        indices = np.argsort(self.position)[::-1]
        self.position = np.zeros_like(self.position)
        self.position[indices[:8]] = 1  # n_select changed to 8


def load_stockwell_data(file_path):
    """Load Stockwell transformed EEG data"""
    try:
        data = np.load(file_path)
        print(f"Loaded data with shape: {data.shape}")
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


def find_label_file(stockwell_file):
    """Find the corresponding label file for a Stockwell file"""
    base_dir = os.path.dirname(stockwell_file)
    base_name = os.path.basename(stockwell_file)

    # Extract the pattern from the stockwell filename
    # Expected pattern: _stockwell_[type]_[duration]s_[freq]Hz.npy
    match = re.search(r'_stockwell_(.+?)_(\d+)s_(\d+)Hz\.npy', base_name)
    if not match:
        return None

    segment_type = match.group(1)
    duration = match.group(2)

    # Try to find matching label file
    # Expected pattern: _labels_[type]_[duration]s.npy
    label_pattern = f"_labels_{segment_type}_{duration}s.npy"

    # For seizure files which might have special naming
    if "seizure" in segment_type and segment_type != "seizure":
        # Handle pre-seizure, post-seizure
        label_pattern = f"_labels_{segment_type}_{duration}s.npy"
    elif "seizure" in segment_type:
        # Extract seizure number if present
        seizure_match = re.search(r'seizure_(\d+)', segment_type)
        if seizure_match:
            seizure_num = seizure_match.group(1)
            label_pattern = f"_labels_seizure_{seizure_num}.npy"
        else:
            label_pattern = f"_labels_seizure.npy"

    # Look for the label file in the same directory
    for file in os.listdir(base_dir):
        if label_pattern in file:
            return os.path.join(base_dir, file)

    # If no specific match found, try a more general approach
    for file in os.listdir(base_dir):
        if f"_labels_{segment_type}" in file:
            return os.path.join(base_dir, file)

    return None


def compute_functional_connectivity(data, method='correlation'):
    """Compute functional connectivity matrix between channels"""
    n_epochs, n_channels, n_freq, n_times = data.shape

    # Reshape data for easier processing
    reshaped_data = np.mean(data, axis=2)  # Average across frequencies
    reshaped_data = reshaped_data.reshape(n_epochs, n_channels, -1)

    # Calculate connectivity matrix
    connectivity = np.zeros((n_channels, n_channels))

    for ch1 in range(n_channels):
        for ch2 in range(n_channels):
            if ch1 == ch2:
                connectivity[ch1, ch2] = 1.0
                continue

            # Correlation across all epochs
            correlations = []
            for epoch in range(n_epochs):
                if method == 'correlation':
                    corr = np.corrcoef(reshaped_data[epoch, ch1], reshaped_data[epoch, ch2])[0, 1]
                    correlations.append(corr)
                elif method == 'mutual_info':
                    # Discretize for mutual information
                    bins = 10
                    c_xy = np.histogram2d(reshaped_data[epoch, ch1], reshaped_data[epoch, ch2], bins)[0]
                    mi = mutual_info_score(None, None, contingency=c_xy)
                    correlations.append(mi)

            connectivity[ch1, ch2] = np.mean(correlations)

    return connectivity


def compute_channel_importance(data):
    """计算通道重要性，替代Granger因果关系分析"""
    n_epochs, n_channels, n_freq, n_times = data.shape

    # 初始化通道重要性得分
    importance_scores = np.zeros(n_channels)

    # 计算每个通道的平均能量
    energy = np.zeros(n_channels)
    for ch in range(n_channels):
        # 平均所有epochs和频率
        ch_data = np.mean(np.abs(data[:, ch, :, :]), axis=(0, 1))
        energy[ch] = np.sum(ch_data ** 2) / len(ch_data)

    # 计算每个通道的频域熵
    entropy = np.zeros(n_channels)
    for ch in range(n_channels):
        # 取平均频谱
        avg_spectrum = np.mean(np.mean(data[:, ch, :, :], axis=0), axis=1)
        # 计算归一化功率谱
        power = np.abs(avg_spectrum) ** 2
        power_norm = power / (np.sum(power) + 1e-10)
        # 计算熵
        power_norm = power_norm[power_norm > 0]  # 避免log(0)
        entropy[ch] = -np.sum(power_norm * np.log2(power_norm))

    # 计算通道间的相关性 - 高相关性表示信息冗余
    redundancy = np.zeros(n_channels)
    avg_data = np.mean(data, axis=(0, 2))  # 平均所有epochs和频率
    for ch1 in range(n_channels):
        ch1_correlations = []
        for ch2 in range(n_channels):
            if ch1 != ch2:
                corr = np.abs(np.corrcoef(avg_data[ch1], avg_data[ch2])[0, 1])
                ch1_correlations.append(corr)
        redundancy[ch1] = np.mean(ch1_correlations)

    # 计算最终重要性得分：高能量、高熵、低冗余的通道更重要
    importance_scores = 0.4 * (energy / np.max(energy)) + \
                        0.4 * (entropy / np.max(entropy)) - \
                        0.2 * (redundancy / np.max(redundancy))

    return importance_scores


def compute_betweenness_centrality(connectivity):
    """Compute betweenness centrality from connectivity matrix"""
    n = connectivity.shape[0]

    # Threshold and binarize connectivity
    threshold = np.percentile(connectivity, 70)  # Top 30% connections
    adj_matrix = (connectivity > threshold).astype(float)
    np.fill_diagonal(adj_matrix, 0)  # No self-connections

    # Initialize centrality values
    betweenness = np.zeros(n)

    # Simple implementation of betweenness centrality
    # For each node pair, find shortest paths and count node occurrences
    for s in range(n):
        for t in range(s + 1, n):
            # Find shortest paths from s to t using breadth-first search
            visited = np.zeros(n, dtype=bool)
            distance = np.ones(n) * np.inf
            distance[s] = 0
            queue = [s]
            predecessors = [[] for _ in range(n)]

            while queue:
                node = queue.pop(0)
                if node == t:
                    break

                if visited[node]:
                    continue

                visited[node] = True
                neighbors = np.where(adj_matrix[node] > 0)[0]

                for neighbor in neighbors:
                    if distance[neighbor] > distance[node] + 1:
                        distance[neighbor] = distance[node] + 1
                        predecessors[neighbor] = [node]
                        queue.append(neighbor)
                    elif distance[neighbor] == distance[node] + 1:
                        predecessors[neighbor].append(node)
                        if neighbor not in queue:
                            queue.append(neighbor)

            # Count paths through each node
            if distance[t] != np.inf:
                # Count shortest paths
                n_paths = np.zeros(n)
                n_paths[s] = 1
                nodes_by_distance = sorted([(distance[i], i) for i in range(n) if distance[i] < np.inf])

                for _, node in nodes_by_distance:
                    for pred in predecessors[node]:
                        n_paths[node] += n_paths[pred]

                # Calculate dependency
                dependency = np.zeros(n)
                nodes_by_reverse_distance = sorted([(distance[i], i) for i in range(n) if distance[i] < np.inf],
                                                   reverse=True)

                for _, node in nodes_by_reverse_distance:
                    if node != s and node != t:
                        for pred in predecessors[node]:
                            if n_paths[node] > 0:
                                dependency[pred] += (n_paths[pred] / n_paths[node]) * (1 + dependency[node])

                betweenness += dependency

    return betweenness


def evaluate_channel_subset(data, channel_mask, labels=None):
    """Evaluate a subset of channels based on signal characteristics"""
    selected_data = data[:, channel_mask.astype(bool), :, :]

    # If labels are available, use classification accuracy
    if labels is not None and len(labels) == data.shape[0]:
        # 修复：处理样本数量为1的情况
        if len(labels) == 1:
            # 如果只有一个样本，使用信号特征进行评估
            variances = np.var(np.mean(selected_data, axis=2), axis=2).mean()
            spectral_entropy = compute_spectral_entropy(selected_data)
            channel_count_score = 1 - abs(np.sum(channel_mask) - 8) / 8  # n_select changed to 8

            # Combined score
            score = 0.4 * variances + 0.4 * spectral_entropy + 0.2 * channel_count_score
            return score

        # 正常情况下的分类评估
        variances = np.var(np.mean(selected_data, axis=2), axis=2)  # Variance across time
        X_train, X_test, y_train, y_test = train_test_split(variances, labels, test_size=0.3, random_state=42)

        # Simple threshold classifier
        best_accuracy = 0
        best_threshold = 0
        for threshold in np.linspace(np.min(X_train), np.max(X_train), 20):
            predictions = (X_train.mean(axis=1) > threshold).astype(int)
            accuracy = np.mean(predictions == y_train)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_threshold = threshold

        # Evaluate on test set
        test_predictions = (X_test.mean(axis=1) > best_threshold).astype(int)
        accuracy = np.mean(test_predictions == y_test)
        return accuracy

    # Without labels, use signal characteristics
    # 1. Signal variance (higher is better)
    variances = np.var(np.mean(selected_data, axis=2), axis=2).mean()

    # 2. Spectral entropy
    spectral_entropy = compute_spectral_entropy(selected_data)

    # 3. Optimal number of channels (closer to n_select is better)
    channel_count_score = 1 - abs(np.sum(channel_mask) - 8) / 8  # n_select changed to 8

    # Combined score
    score = 0.4 * variances + 0.4 * spectral_entropy + 0.2 * channel_count_score
    return score


def compute_spectral_entropy(data):
    """计算光谱熵 - 单独函数方便重用"""
    spectral_entropy = 0
    n_epochs, n_channels = data.shape[0], data.shape[1]

    for epoch in range(min(n_epochs, 5)):  # Limit to 5 epochs for speed
        for ch in range(n_channels):
            # Average over frequencies
            signal_avg = np.mean(data[epoch, ch], axis=0)
            ps = np.abs(np.fft.fft(signal_avg)) ** 2
            ps_norm = ps / (np.sum(ps) + 1e-10)
            ps_norm = ps_norm[ps_norm > 0]  # Remove zeros
            if len(ps_norm) > 0:  # Ensure there are non-zero values
                spectral_entropy += -np.sum(ps_norm * np.log2(ps_norm)) / np.log2(len(ps_norm))

    spectral_entropy /= (min(n_epochs, 5) * n_channels)
    return spectral_entropy


def quantum_pso_channel_selection(data, centrality_scores, importance_scores, n_select=8, n_particles=10, iterations=20,
                                  labels=None):
    """Select channels using Quantum PSO"""
    n_channels = data.shape[1]

    # Initialize particles
    particles = [QPSOParticle(n_channels, n_select) for _ in range(n_particles)]
    gbest = np.zeros(n_channels)
    gbest_fitness = -float('inf')

    # Initialize particles with some bias from centrality and importance scores
    for i, particle in enumerate(particles):
        # Add bias toward high centrality and importance channels
        combined_score = 0.5 * centrality_scores / (centrality_scores.max() + 1e-10) + 0.5 * importance_scores / (
                    importance_scores.max() + 1e-10)
        top_indices = np.argsort(combined_score)[-n_select:]
        particle.position = np.zeros(n_channels)
        particle.position[top_indices] = 1

        # Evaluate initial position
        fitness = evaluate_channel_subset(data, particle.position, labels)
        particle.pbest_fitness = fitness

        if fitness > gbest_fitness:
            gbest_fitness = fitness
            gbest = particle.position.copy()

    # Main QPSO loop
    history = []
    for t in range(iterations):
        beta = 1.0 - 0.5 * t / iterations  # Linear decreasing beta

        for particle in particles:
            # Update position
            particle.update_position(gbest, beta)

            # Evaluate fitness
            fitness = evaluate_channel_subset(data, particle.position, labels)

            # Update personal best
            if fitness > particle.pbest_fitness:
                particle.pbest_fitness = fitness
                particle.pbest = particle.position.copy()

            # Update global best
            if fitness > gbest_fitness:
                gbest_fitness = fitness
                gbest = particle.position.copy()

        history.append(gbest_fitness)
        print(f"Iteration {t + 1}/{iterations}, Best fitness: {gbest_fitness:.4f}, Selected channels: {np.sum(gbest)}")

    # Get final selected channels
    selected_channels = np.where(gbest == 1)[0]

    # If not exactly n_select channels, force it
    if len(selected_channels) != n_select:
        combined_score = 0.5 * centrality_scores / (centrality_scores.max() + 1e-10) + 0.5 * importance_scores / (
                    importance_scores.max() + 1e-10)
        if len(selected_channels) < n_select:
            # Add more channels
            remaining = list(set(range(n_channels)) - set(selected_channels))
            sorted_remaining = sorted(remaining, key=lambda x: combined_score[x], reverse=True)
            selected_channels = np.append(selected_channels, sorted_remaining[:n_select - len(selected_channels)])
        else:
            # Remove excess channels
            sorted_channels = sorted(selected_channels, key=lambda x: combined_score[x])
            selected_channels = np.array(sorted_channels[len(selected_channels) - n_select:])

    # Calculate final channel rankings
    rankings = np.zeros(n_channels)
    rankings[selected_channels] = range(n_select, 0, -1)  # Higher rank is better (n_select is best)

    return selected_channels, rankings, history


def process_and_select_channels(input_file, n_select=8):
    """Process Stockwell data and select the best channels"""
    print(f"\nProcessing file: {os.path.basename(input_file)}")

    # 设置输出文件名
    base_name = os.path.basename(input_file)
    dir_name = os.path.dirname(input_file)
    output_file = os.path.join(dir_name, base_name.replace('.npy', f'_selected{n_select}ch.npy'))

    # Load data
    data = load_stockwell_data(input_file)
    if data is None:
        return

    # Load labels if available
    label_file = find_label_file(input_file)
    labels = None
    if label_file and os.path.exists(label_file):
        try:
            labels = np.load(label_file)
            print(f"Loaded {len(labels)} labels (from {os.path.basename(label_file)})")
        except Exception as e:
            print(f"Error loading labels: {e}")

    n_epochs, n_channels, n_freq, n_times = data.shape
    print(f"Processing data: {n_epochs} epochs, {n_channels} channels")

    # 1. Compute functional connectivity
    print("Computing functional connectivity...")
    connectivity = compute_functional_connectivity(data)

    # 2. Compute centrality measures
    print("Computing betweenness centrality...")
    centrality = compute_betweenness_centrality(connectivity)

    # 3. Compute channel importance (replacement for Granger causality)
    print("Computing channel importance...")
    importance_scores = compute_channel_importance(data)

    # 4. Select channels using QPSO
    print("Selecting channels using QPSO...")
    selected_channels, channel_rankings, history = quantum_pso_channel_selection(
        data, centrality, importance_scores, n_select=n_select, labels=labels
    )

    # Print information about selected channels and rankings
    print("\n----- CHANNEL RANKINGS -----")
    # Sort channels by ranking for display
    channel_ranking_pairs = [(i, channel_rankings[i]) for i in range(n_channels)]
    sorted_channels = sorted(channel_ranking_pairs, key=lambda x: x[1], reverse=True)

    # Print all channels sorted by importance
    print(f"{'Channel':<8}{'Rank':<8}{'Centrality':<12}{'Importance':<12}{'Selected':<8}")
    print("-" * 48)

    for ch_idx, rank in sorted_channels:
        is_selected = "Yes" if ch_idx in selected_channels else "No"
        print(
            f"{ch_idx:<8}{int(rank) if rank > 0 else '-':<8}{centrality[ch_idx]:<12.4f}{importance_scores[ch_idx]:<12.4f}{is_selected:<8}")

    print("\n----- SELECTED CHANNELS SUMMARY -----")
    print(f"Selected {len(selected_channels)} channels: {', '.join(map(str, selected_channels))}")

    # Extract selected channel data
    selected_data = data[:, selected_channels, :, :]

    # Save selected data
    np.save(output_file, selected_data)
    print(f"Selected channel data saved to {output_file}")

    # Visualize results
    plt.figure(figsize=(12, 10))

    # Plot connectivity matrix
    plt.subplot(2, 2, 1)
    plt.imshow(connectivity, cmap='viridis')
    plt.colorbar()
    plt.title("Functional Connectivity Matrix")

    # Plot centrality
    plt.subplot(2, 2, 2)
    plt.bar(range(n_channels), centrality)
    plt.axhline(y=np.median(centrality), color='r', linestyle='--')
    for ch in selected_channels:
        plt.bar(ch, centrality[ch], color='red')
    plt.title("Betweenness Centrality")
    plt.xlabel("Channel")

    # Plot importance scores
    plt.subplot(2, 2, 3)
    plt.bar(range(n_channels), importance_scores)
    plt.axhline(y=np.median(importance_scores), color='r', linestyle='--')
    for ch in selected_channels:
        plt.bar(ch, importance_scores[ch], color='red')
    plt.title("Channel Importance")
    plt.xlabel("Channel")

    # Plot QPSO convergence
    plt.subplot(2, 2, 4)
    plt.plot(range(1, len(history) + 1), history)
    plt.title("QPSO Convergence")
    plt.xlabel("Iteration")
    plt.ylabel("Fitness")

    # Save the figure
    plt.tight_layout()
    visualization_file = output_file.replace('.npy', '_visualization.png')
    plt.savefig(visualization_file)
    print(f"Visualization saved to {visualization_file}")
    plt.close()

    return output_file


def process_all_stockwell_files(data_dir, n_select=8):
    """Process all Stockwell files in the directory"""
    print(f"Searching for Stockwell transform files in {data_dir}...")
    processed_files = []

    # Find all Stockwell transformed files
    stockwell_files = []
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if "_stockwell_" in file and file.endswith(".npy") and not "selected" in file:
                stockwell_files.append(os.path.join(root, file))

    print(f"Found {len(stockwell_files)} Stockwell transform files")

    # Process each file
    for i, file in enumerate(stockwell_files):
        print(f"\nProcessing file {i + 1}/{len(stockwell_files)}: {os.path.basename(file)}")
        try:
            output_file = process_and_select_channels(file, n_select=n_select)
            if output_file:
                processed_files.append(output_file)
        except Exception as e:
            print(f"Error processing file {file}: {e}")
            import traceback
            traceback.print_exc()

    print(f"\nProcessing complete! Processed {len(processed_files)} files")
    return processed_files


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="EEG Channel Selection for Stockwell-transformed data")
    parser.add_argument("--data_dir", "-d", default="data/eeg_data",
                        help="Directory containing Stockwell transform data")
    parser.add_argument("--channels", "-c", type=int, default=8, help="Number of channels to select (default: 8)")

    args = parser.parse_args()

    print(f"Starting selection of {args.channels} most important channels from {args.data_dir}...")
    process_all_stockwell_files(args.data_dir, n_select=args.channels)